**Modelo clasificador de imagenes generadas con DeepFake vs Reales**

*Dataset obtenido de:* https://www.kaggle.com/datasets/muhammadbilal6305/200k-real-vs-ai-visuals-by-mbilal/data

**Preprocesamiento**

In [1]:
import os
import numpy as np
import pandas as pd

In [2]:
dataset_path = "./dataset/"
images_dir_path = dataset_path + "/images"

In [3]:
train_df = pd.read_csv(dataset_path + "train_labels.csv")
val_df = pd.read_csv(dataset_path + "val_labels.csv")
test_df = pd.read_csv(dataset_path + "test_labels.csv")
train_df.head()

,Unnamed: 0,filename,label,resolution,image_size
0,0,03428.jpg,1,256x256,65536
1,1,38472.jpg,1,256x256,65536
2,2,49600.jpg,1,256x256,65536
3,3,68452.jpg,1,256x256,65536
4,4,60461.jpg,1,256x256,65536


In [4]:
train_df.loc[train_df["label"] == 0, "full_path"] = images_dir_path + '/ai_images/' + train_df["filename"]
train_df.loc[train_df["label"] == 1, "full_path"] = images_dir_path + '/real/' + train_df["filename"]
val_df.loc[train_df["label"] == 0, "full_path"] = images_dir_path + '/ai_images/' + train_df["filename"]
val_df.loc[train_df["label"] == 1, "full_path"] = images_dir_path + '/real/' + train_df["filename"]
test_df.loc[train_df["label"] == 0, "full_path"] = images_dir_path + '/ai_images/' + train_df["filename"]
test_df.loc[train_df["label"] == 1, "full_path"] = images_dir_path + '/real/' + train_df["filename"]


In [5]:
train_paths = train_df['full_path']
train_labels = train_df['label']
val_paths = val_df['full_path']
val_labels = val_df['label']
test_paths = test_df['full_path']
test_labels = test_df['label']

In [6]:
import tensorflow as tf

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

def load_and_preprocess_image(path, label):
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, IMG_SIZE)
    image = image / 255.0
    return image, label

def create_dataset(paths, labels, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(load_and_preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(buffer_size=1000)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

In [7]:
train_ds = create_dataset(train_paths, train_labels, shuffle=True)
val_ds = create_dataset(val_paths, val_labels)
test_ds = create_dataset(test_paths, test_labels)

**Entrenamiento**

In [9]:
from tensorflow import keras
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 100

model = keras.Sequential([

    keras.layers.Input(shape=(*IMG_SIZE, 3)),

    keras.layers.Conv2D(32, (3, 3), activation='relu'),
    keras.layers.BatchNormalization(),
    keras.layers.MaxPooling2D((2, 2)),

    keras.layers.Conv2D(64, (3, 3), activation='relu'),
    keras.layers.BatchNormalization(),

    keras.layers.MaxPooling2D((2, 2)),
    keras.layers.BatchNormalization(),
    keras.layers.Conv2D(128, (3, 3), activation='relu'),
    keras.layers.BatchNormalization(),
    keras.layers.MaxPooling2D((2, 2)),

    keras.layers.Dropout(0.3),

    keras.layers.Flatten(),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dense(1, activation='sigmoid')
])

early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

model_checkpoint = ModelCheckpoint(
    'best_model.keras',
    monitor='val_accuracy',
    save_best_only=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,
    patience=5,
    min_lr=1e-7
  )

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
  )

history = model.fit(
    train_ds,
    epochs=EPOCHS,
    validation_data=val_ds,
    callbacks=[early_stopping, model_checkpoint, reduce_lr],
    verbose=1
)

Epoch 1/100


: 

In [ ]:
model.save("best_model.keras")

In [ ]:
results = model.evaluate(val_ds)
print(results)

In [ ]:
y_pred = model.predict(val_ds)
y_pred = (y_pred > 0.5).astype(int)